# Phase 3 — Step 3.1: Model Evaluation

Loads the best Keras checkpoint and produces a full evaluation report:
1. Confusion matrix heatmap
2. Per-class precision, recall, F1
3. Top-k accuracy (does the correct sign appear in the top 3 predictions?)
4. Confidence distribution and threshold analysis
5. Saves all results to `/content/drive/MyDrive/nsl-evaluation/`

## Cell 1 — Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from sklearn.model_selection import train_test_split

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")

**What each metric means:**

- **Precision** — out of all the times the model said class X, how often was it right?  High precision = few false positives.
- **Recall** — out of all the actual class X samples, how many did the model find?  High recall = few false negatives.
- **F1 score** — the harmonic mean of precision and recall.  Best metric when classes are imbalanced.  F1 = 1 is perfect, F1 = 0 is useless.
- **Top-3 accuracy** — was the correct answer in the model's top 3 guesses?  Important for real apps where the user picks the right one from a short list.

## Cell 2 — Mount Drive and load model + data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/nsl-data'
MODEL_PATH = '/content/drive/MyDrive/nsl-checkpoints/best_model.h5'
EVAL_DIR = '/content/drive/MyDrive/nsl-evaluation'
os.makedirs(EVAL_DIR, exist_ok=True)

X = np.load(os.path.join(DATA_DIR, 'X.npy'))
y = np.load(os.path.join(DATA_DIR, 'y.npy'))
with open(os.path.join(DATA_DIR, 'label_map.json')) as f:
    label_map = json.load(f)
idx_to_label = {v: k for k, v in label_map.items()}
n_classes = len(label_map)

# Rebuild the same test split the training notebook used
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

# Load the best model
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

print(f"\nTest set size: {X_test.shape[0]}")
print(f"Classes: {n_classes}")

## Cell 3 — Predict on the test set

In [ ]:
# Get probability distribution over all classes for each test sample
y_proba = model.predict(X_test, verbose=0)
y_pred  = y_proba.argmax(axis=1)

print(f"Predictions shape: {y_proba.shape}")
print(f"  N samples: {y_proba.shape[0]}")
print(f"  N classes: {y_proba.shape[1]}")

# Top-k accuracy (how often is the correct answer in the top 3?)
top1 = (y_pred == y_test).mean()
top3 = top_k_accuracy_score(y_test, y_proba, k=3, labels=range(n_classes))
print(f"\nTop-1 accuracy: {top1:.4f}  ({top1*100:.1f}%)")
print(f"Top-3 accuracy: {top3:.4f}  ({top3*100:.1f}%)")

# Save raw predictions for later cells
np.save(os.path.join(EVAL_DIR, 'y_test.npy'),  y_test)
np.save(os.path.join(EVAL_DIR, 'y_pred.npy'),  y_pred)
np.save(os.path.join(EVAL_DIR, 'y_proba.npy'), y_proba)

## Cell 4 — Confusion matrix heatmap

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=list(range(n_classes)))
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_norm = np.nan_to_num(cm_norm)  # 0/0 -> 0

class_names = [idx_to_label[i] for i in range(n_classes)]
fig, ax = plt.subplots(figsize=(max(8, n_classes * 0.7),
                            max(6, n_classes * 0.6)))
sns.heatmap(cm_norm, annot=True, fmt=".2f",
            xticklabels=class_names, yticklabels=class_names,
            cmap="Blues", ax=ax, cbar_kws={"label": "Recall"},
            annot_kws={"size": 10})
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix  (rows = true, columns = predicted)\n"
             f"Normalised so each row sums to 1.0")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'confusion_matrix.png'),
            dpi=120, bbox_inches='tight')
plt.show()

print("Saved confusion_matrix.png")
print("\nHow to read this:")
print("  • Diagonal = correct predictions (higher = better)")
print("  • Off-diagonal = mistakes.  Each cell shows the fraction of true")
print("    class samples that were misclassified as some other class.")
print("  • If two signs have high values off-diagonal between them, those")
print("    two signs look too similar in the data and the model confuses them.")

## Cell 5 — Per-class precision / recall / F1

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=list(range(n_classes)))

per_class_df = pd.DataFrame({
    "class":     class_names,
    "support":   support,
    "precision": np.round(precision, 3),
    "recall":    np.round(recall, 3),
    "f1":        np.round(f1, 3),
}).sort_values("f1", ascending=False)

print(per_class_df.to_string(index=False))
per_class_df.to_csv(os.path.join(EVAL_DIR, 'per_class_metrics.csv'),
                    index=False)
print(f"\nSaved per_class_metrics.csv")

# Bar chart of F1 scores
fig, ax = plt.subplots(figsize=(10, max(4, n_classes * 0.4)))
sorted_df = per_class_df.sort_values("f1")
bars = ax.barh(sorted_df["class"], sorted_df["f1"],
               color=sns.color_palette("viridis", n_classes))
ax.set_xlabel("F1 score")
ax.set_title("F1 score per class  (sorted, higher = better)")
ax.set_xlim(0, 1.05)
for bar, f1_val in zip(bars, sorted_df["f1"]):
    ax.text(f1_val + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{f1_val:.2f}",
            va="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'per_class_f1.png'),
            dpi=120, bbox_inches='tight')
plt.show()

## Cell 6 — Confidence distribution

In [ ]:
# confidence = max probability assigned by the model
y_conf = y_proba.max(axis=1)
correct = (y_pred == y_test)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_conf[correct],   bins=20, alpha=0.7,
        label=f"Correct ({(correct).sum()} samples)",
        color="#22c55e")
ax.hist(y_conf[~correct],  bins=20, alpha=0.7,
        label=f"Wrong ({(~correct).sum()} samples)",
        color="#ef4444")
ax.set_xlabel("Model confidence  (max predicted probability)")
ax.set_ylabel("Number of test samples")
ax.set_title("Confidence distribution by correctness")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'confidence_distribution.png'),
            dpi=120, bbox_inches='tight')
plt.show()

print(f"Mean confidence (correct): {y_conf[correct].mean():.3f}")
print(f"Mean confidence (wrong)  : {y_conf[~correct].mean():.3f}")
print("\nHow to read this:")
print("  • If the red bars cluster at low confidence and green at high,")
print("    the model 'knows what it doesn't know' — useful for production.")
print("  • If wrong predictions also have high confidence, the model is")
print("    confidently wrong, which is the dangerous failure mode.")

## Cell 7 — Threshold analysis

In [ ]:
# How accuracy changes if we only accept predictions above a confidence threshold
thresholds = np.arange(0.0, 1.01, 0.05)
rows = []
for t in thresholds:
    keep = y_conf >= t
    if keep.sum() == 0:
        continue
    acc_kept = (y_pred[keep] == y_test[keep]).mean()
    coverage = keep.mean()
    rows.append({"threshold": t,
                 "coverage": coverage,
                 "accuracy": acc_kept,
                 "n_kept": int(keep.sum())})
thresh_df = pd.DataFrame(rows)
print(thresh_df.round(3).to_string(index=False))
thresh_df.to_csv(os.path.join(EVAL_DIR, 'threshold_analysis.csv'),
                 index=False)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(thresh_df["threshold"], thresh_df["accuracy"],
         "o-", color="#3b82f6", label="Accuracy on kept predictions")
ax1.set_xlabel("Confidence threshold")
ax1.set_ylabel("Accuracy", color="#3b82f6")
ax1.tick_params(axis="y", labelcolor="#3b82f6")
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(thresh_df["threshold"], thresh_df["coverage"],
         "s--", color="#f59e0b", label="Coverage (fraction kept)")
ax2.set_ylabel("Coverage", color="#f59e0b")
ax2.tick_params(axis="y", labelcolor="#f59e0b")
ax2.set_ylim(0, 1.05)

plt.title("Threshold analysis\n"
          "(higher threshold = fewer but more accurate predictions)")
fig.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'threshold_analysis.png'),
            dpi=120, bbox_inches='tight')
plt.show()

print("\nHow to use this:")
print("  • Pick a threshold where coverage is still reasonable (>= 0.5)")
print("    but accuracy is high.  That is your production confidence gate.")
print("  • Predictions below that threshold get marked 'uncertain'")
print("    instead of being shown to the user as a final answer.")

## Cell 8 — Full classification report (text file)

In [ ]:
report = classification_report(
    y_test, y_pred,
    labels=list(range(n_classes)),
    target_names=class_names,
    zero_division=0,
    digits=3,
)
print(report)

report_path = os.path.join(EVAL_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(f"NSL Sign Translator - Test Set Evaluation\n")
    f.write(f"============================================\n\n")
    f.write(f"Total test samples : {len(y_test)}\n")
    f.write(f"Total classes      : {n_classes}\n")
    f.write(f"Top-1 accuracy     : {top1:.4f}\n")
    f.write(f"Top-3 accuracy     : {top3:.4f}\n\n")
    f.write(report)
print(f"\nSaved {report_path}")

## Cell 9 — Top confused pairs

In [ ]:
# Off-diagonal cells with the highest misclassification rate
n = min(10, n_classes * (n_classes - 1))
cm_offdiag = cm_norm.copy()
np.fill_diagonal(cm_offdiag, 0)  # ignore the diagonal (correct predictions)
flat = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm_offdiag[i, j] > 0:
            flat.append((cm_offdiag[i, j], i, j))
flat.sort(reverse=True)
flat = flat[:n]

print(f"Top {len(flat)} confused sign pairs:")
print(f"{'True':<20s}  {'Predicted':<20s}  {'Fraction':>8s}  {'Count':>5s}")
for frac, i, j in flat:
    count = cm[i, j]
    print(f"{idx_to_label[i]:<20s}  {idx_to_label[j]:<20s}  "
          f"{frac:>8.2%}  {count:>5d}")

if flat:
    print("\nInterpretation:")
    print("  These are the sign pairs the model gets wrong most often.")
    print("  In production these would need either:")
    print("    (a) more training data for both signs")
    print("    (b) better annotations (cleaner IN/OUT boundaries)")
    print("    (c) re-recording the source video with better contrast")

## Cell 10 — Summary of all generated artefacts

In [ ]:
print("=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)
print(f"Test top-1 accuracy: {top1*100:.1f}%")
print(f"Test top-3 accuracy: {top3*100:.1f}%")
print()
print(f"All artefacts saved to {EVAL_DIR}:")
for fname in sorted(os.listdir(EVAL_DIR)):
    size = os.path.getsize(os.path.join(EVAL_DIR, fname)) / 1024
    print(f"  {fname:<30s}  {size:>8.1f} KB")
print()
print("Next steps:")
print("  1. Review the confusion matrix to see which signs are confused")
print("  2. Open 04_tflite_conversion.ipynb to export the mobile model")